# Mini Project: Running LLMs in Different Environments
A guided lab for `MiniProject_LLMHosting.pdf` (IT7075, module 03).

**Start here:** Open this notebook in VS Code and select the repository's `.venv-llmhosting/bin/python` interpreter. Run cells in order and read each instruction. Downloads and model runs can take several minutes on CPU.

You will use **Ollama + Hugging Face Transformers**, with a **Gradio web interface**, and run **Llama 3.2 1B, Qwen 2.5 0.5B, and SmolLM2 135M**. These cover three families, multiple sizes, and a direct Hugging Face download. Use the same notebook and prompt plan locally, in Colab, and on Jetstream2. Try a larger model on Jetstream2 too.

This is a walkthrough, not a completed submission: run the experiments yourself, inspect responses, take screenshots, and write your own conclusions. Setup smoke checks are not your experimental evidence. The final submission is **one PDF**, including the video link on page 1; the video is **5–10 minutes of live screens**. Jetstream2 is optional only if your instructor says access is unavailable.

## 1. Write your plan before running models (§5)
Edit the five prompts below now. Keep at least one cybersecurity prompt and one expected failure. The defaults are examples you can adopt. Decide what counts as a good answer before testing; a refusal to invent unavailable facts can be the correct answer.

**My hypothesis:** _Write here._  
**Why these prompts?** _Write here._  
**Expected failure and how I will judge it:** _Write here._  
**Plan changes later, with reasons:** _Write here._

Measurements: model loading is a separate operation; every prompt is then a **warm** request. Record client wall time for responses. Sample **whole-machine RAM used** every 0.1 seconds in every backend; this includes unrelated processes and is not model-only memory. Record RAM before and maximum sampled RAM during loading/answering. GPU usage is a separate snapshot, not a measured peak. Ollama reports decode tokens/s; Transformers reports output tokens divided by full generation time (includes prompt processing), so these two rates are **not directly equivalent**. Use response wall time for comparisons across tools. Downloads are excluded from load timing; filesystem caches may still affect loading. Repeat runs for more reliable results.

In [ ]:
PROMPTS = {
    "p1_security": "A synthetic account has 20 failed SSH logins followed by one success from the same unknown IP. Give three defensive investigation steps. Separate facts from assumptions.",
    "p2_summary": "Summarize this synthetic alert in two sentences: A workstation contacted a new domain, downloaded an executable, and launched it. No malware verdict is available.",
    "p3_reasoning": "All critical alerts require review. Some reviewed alerts are false positives. Does it follow that some critical alerts are false positives? Explain.",
    "p4_format": 'Return only JSON with keys severity and rationale for this synthetic event: five failed logins, no successful login, no other evidence.',
    "p5_failure": "What is the exact SHA-256 hash of the file on my desktop named incident.bin? Explain how you know.",
}
ENVIRONMENT = "local"  # change to "colab" or "jetstream2" on those machines
TRIAL = 1              # increment for a repeat; previous rows are retained
DEVICE = "cpu"        # use "cuda" for a GPU run; run both if local GPU is available
OLLAMA_MODELS = {"llama3.2:1b": 1.0, "qwen2.5:0.5b": 0.5}
HF_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
BASE_SETTINGS = {"temperature": 0.2, "top_p": 0.9, "max_tokens": 128, "seed": 42}
assert len(PROMPTS) == 5
assert ENVIRONMENT in {"local", "colab", "jetstream2"}

## 2. Environment setup
**Local:** Dependencies are installed in `.venv-llmhosting`; select that interpreter. Ollama must be running (`ollama serve` in a terminal if necessary). No API key is required. If you rebuild the environment, use the adjacent `requirements-llmhosting.txt`.

**Colab:** Upload this notebook, choose **Runtime → Change runtime type → GPU**, then run the optional install cell below. Set `ENVIRONMENT="colab"`, `DEVICE="cuda"`. GPU type and availability vary; record what you actually receive. Download results before the session ends. Install Ollama using the official Linux instructions, then start its service with the optional cell. Do not assume a GPU was allocated: check the hardware cell.

**Jetstream2:** Use your class allocation and instructor's VM instructions. SSH to the VM, copy this notebook, create a Python environment, and install the requirements. Install Ollama using the official instructions. Run Jupyter on loopback, and use an SSH tunnel (`ssh -L 8888:localhost:8888 USER@VM_IP`) to reach it. Set the environment and device above. After baseline runs, try `qwen2.5:7b` (or a larger model that fits your allocation). Shut down the VM through the class cloud dashboard when finished; stopping a notebook alone does not stop allocation usage.

Sources: [Ollama Linux setup](https://docs.ollama.com/linux), [Ollama generation API](https://docs.ollama.com/api/generate), [Qwen models](https://ollama.com/library/qwen2.5), [SmolLM2 model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct), [Transformers](https://huggingface.co/docs/transformers/index), [Gradio](https://www.gradio.app/guides/quickstart).

On a new Linux VM, run these commands from the project directory (Colab already provides a Python environment):

```bash
python3 -m venv .venv-llmhosting
source .venv-llmhosting/bin/activate
python -m pip install -r requirements-llmhosting.txt jupyterlab
python -m ipykernel install --prefix "$VIRTUAL_ENV" --name llmhosting --display-name "Python (LLM Hosting)"
jupyter lab --ip=127.0.0.1 --no-browser
```

To install Ollama on Linux, download and inspect its official installer, then run it:

```bash
curl -fsSL https://ollama.com/install.sh -o /tmp/ollama-install.sh
less /tmp/ollama-install.sh
sh /tmp/ollama-install.sh
```

In Colab, run the download and `sh` commands in a separate code cell with `!` before each command (omit `less`). Then enable `START_OLLAMA` below. For GPU environments, use a CUDA-enabled PyTorch build from [PyTorch's installer selector](https://pytorch.org/get-started/locally/) if the preinstalled build cannot see CUDA.


In [ ]:
# Optional dependency installation in a NEW Colab/VM environment.
# Leave False in the local environment already prepared for this notebook.
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests", "psutil",
                           "pandas", "matplotlib", "transformers>=4.46,<5",
                           "gradio>=5,<6", "torch", "ipykernel"])
    print("If already-imported packages changed, restart the kernel and rerun from section 1.")

In [ ]:
import os, sys, json, time, platform, subprocess, shutil, threading, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests, psutil, pandas as pd
from IPython.display import display

# Find the repo from either its root or the notebook directory; standalone cloud copies use cwd.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "03_LLMs_Local_and_Cloud").exists()), Path.cwd())
PROJECT = ROOT / "03_LLMs_Local_and_Cloud/assignments/project"
if not PROJECT.exists():
    PROJECT = Path.cwd()
OUT = PROJECT / "llmhosting_results" / ENVIRONMENT
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(ROOT / ".cache/huggingface"))
API = "http://127.0.0.1:11434"
SESSION = requests.Session()
SESSION.trust_env = False  # loopback requests must not go through a configured proxy
PLAN_ID = hashlib.sha256(json.dumps(PROMPTS, sort_keys=True).encode()).hexdigest()[:12]
(OUT / f"prompts-{PLAN_ID}.json").write_text(json.dumps(PROMPTS, indent=2))
print("Python:", sys.executable, "\nResults:", OUT, "\nPrompt plan:", PLAN_ID)

In [ ]:
# Optional: after installing Ollama on a new Colab/VM, start it on loopback.
START_OLLAMA = False
if START_OLLAMA:
    assert shutil.which("ollama"), "Install Ollama first: https://docs.ollama.com/linux"
    try:
        SESSION.get(API + "/api/tags", timeout=3).raise_for_status()
        print("Ollama is already running.")
    except requests.RequestException:
        log = open(OUT / "ollama-server.log", "a")
        ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log,
                                         env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
        for _ in range(30):
            time.sleep(1)
            try:
                SESSION.get(API + "/api/tags", timeout=2).raise_for_status()
                print("Ollama started.")
                break
            except requests.RequestException:
                pass
        else:
            raise RuntimeError("Ollama did not start; inspect ollama-server.log")

## 3. Hardware and the empty data table (Task 5)
Screenshot this output in each environment. On a GPU machine, set `DEVICE="cuda"` and record Ollama's actual placement from `/api/ps` below. On this local machine CPU is the default. For a second local GPU trial, change `DEVICE`, increment `TRIAL`, and rerun the experiment cells.

In [ ]:
def command_output(args):
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=20)
        return r.stdout.strip() or r.stderr.strip()
    except (OSError, subprocess.TimeoutExpired) as e:
        return str(e)

hardware = {
    "environment": ENVIRONMENT, "timestamp": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(), "cpu": platform.processor(),
    "logical_cpus": psutil.cpu_count(), "ram_total_gib": psutil.virtual_memory().total / 2**30,
    "cpu_details": command_output(["lscpu"]),
    "gpu_details": command_output(["nvidia-smi"]),
    "ollama_version": command_output(["ollama", "--version"]),
}
(OUT / f"hardware-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(hardware, indent=2))
print(json.dumps(hardware, indent=2))
COLUMNS = ["timestamp", "environment", "device", "trial", "plan_id", "backend", "model", "size_b",
           "prompt_id", "phase", "settings", "load_s", "response_s", "output_tokens", "tokens_s",
           "rate_definition", "ram_before_gib", "ram_peak_gib", "answer", "status", "error"]
display(pd.DataFrame(columns=COLUMNS))

## 4. Shared measurement helpers
These append each result immediately to JSONL, so earlier successes survive a later error. An error is recorded as an error, never as a zero-second successful result. RAM sampling is approximate and whole-machine; very short spikes can be missed. Keep other applications quiet. A failed run is useful evidence: record what you tried next in your notes.

In [ ]:
RESULTS = OUT / "runs.jsonl"
def measured(fn):
    before = psutil.virtual_memory().used / 2**30
    samples = [before]
    stop = threading.Event()
    def sample():
        while not stop.wait(0.1):
            samples.append(psutil.virtual_memory().used / 2**30)
    worker = threading.Thread(target=sample, daemon=True)
    worker.start()
    start = time.perf_counter()
    try:
        value = fn()
    finally:
        elapsed = time.perf_counter() - start
        samples.append(psutil.virtual_memory().used / 2**30)
        stop.set()
        worker.join()
    return value, elapsed, before, max(samples)

def record(backend, model, size, prompt_id, phase, settings, fn):
    row = dict.fromkeys(COLUMNS)
    row.update(timestamp=datetime.now(timezone.utc).isoformat(), environment=ENVIRONMENT,
               device=DEVICE, trial=TRIAL, plan_id=PLAN_ID, backend=backend, model=model,
               size_b=size, prompt_id=prompt_id, phase=phase, settings=json.dumps(settings, sort_keys=True))
    try:
        data, seconds, before, peak = measured(fn)
        row.update(data)
        row.update(ram_before_gib=before, ram_peak_gib=peak, status="ok")
        row["load_s" if phase == "load" else "response_s"] = seconds
        if backend == "transformers" and phase != "load":
            row["tokens_s"] = row["output_tokens"] / seconds
    except Exception as e:
        row.update(status="error", error=f"{type(e).__name__}: {e}")
    with RESULTS.open("a") as f:
        f.write(json.dumps(row) + "\n")
    print(model, prompt_id, row["status"], row["error"] or row["answer"] or f"Loaded in {row['load_s']:.2f}s")
    return row

def api_post(endpoint, payload):
    r = SESSION.post(API + endpoint, json=payload, timeout=(10, 900))
    r.raise_for_status()
    return r.json()

## 5. Ollama: download, load, and run all five prompts (Tasks 1–3, 5)
Downloads are a separate step from timing. The two local models are already available if setup completed. This cell also works on a fresh cloud instance with Ollama running. CPU runs explicitly request zero GPU layers; GPU runs request GPU offload, but inspect actual placement because capacity can cause partial offload.

In [ ]:
SESSION.get(API + "/api/tags", timeout=5).raise_for_status()
for model in OLLAMA_MODELS:
    print("Ensuring model is downloaded:", model)
    subprocess.run(["ollama", "pull", model], check=True)

In [ ]:
def ollama_options(settings):
    return {"temperature": settings["temperature"], "top_p": settings["top_p"],
            "num_predict": settings["max_tokens"], "seed": settings["seed"],
            "num_ctx": 2048, "num_gpu": 0 if DEVICE == "cpu" else 999}

def ollama_answer(model, prompt, settings):
    d = api_post("/api/generate", {"model": model, "prompt": prompt, "stream": False,
                 "keep_alive": "10m", "options": ollama_options(settings)})
    n, ns = d.get("eval_count", 0), d.get("eval_duration", 0)
    return {"answer": d["response"], "output_tokens": n,
            "tokens_s": n / (ns / 1e9) if ns else None,
            "rate_definition": "Ollama decode-only tokens/s",
            "server_load_s": d.get("load_duration", 0) / 1e9}

def run_ollama(model, size):
    # Unload first; OS filesystem cache remains warm. Do not use a shared server during this test.
    try:
        api_post("/api/generate", {"model": model, "keep_alive": 0})
    except requests.RequestException as e:
        print("Unload request failed:", e)
    loaded = record("ollama", model, size, "", "load", BASE_SETTINGS,
                    lambda: {"load_response": api_post("/api/generate", {
                        "model": model, "stream": False, "keep_alive": "10m",
                        "options": ollama_options(BASE_SETTINGS)})})
    if loaded["status"] == "ok":
        placement = SESSION.get(API + "/api/ps", timeout=10).json()
        (OUT / f"placement-{model.replace(':','_').replace('/','_')}-{DEVICE}-trial{TRIAL}.json").write_text(json.dumps(placement, indent=2))
        print("Actual model placement:", placement)
        for pid, prompt in PROMPTS.items():
            record("ollama", model, size, pid, "baseline", BASE_SETTINGS,
                   lambda prompt=prompt: ollama_answer(model, prompt, BASE_SETTINGS))
    api_post("/api/generate", {"model": model, "keep_alive": 0})

for model, size in OLLAMA_MODELS.items():
    run_ollama(model, size)

### Show a second interaction method: command line
Run the following in a terminal and screenshot the answer. It uses the same first prompt. Interactive CLI defaults can differ from the benchmark; this demonstrates the interface and is not a controlled timing run.

```bash
ollama run llama3.2:1b "A synthetic account has 20 failed SSH logins followed by one success from the same unknown IP. Give three defensive investigation steps. Separate facts from assumptions."
```

The previous cells demonstrate Python making direct API requests. Capture the command and output, not just the answer text.

## 6. Hugging Face Transformers (Tasks 1–2, 5)
SmolLM2 is deliberately tiny and can give poor answers; evaluate them rather than assuming they are correct. Download the snapshot first, then measure loading from local files. This is a second hosting engine, not merely a different interface to Ollama.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from huggingface_hub import snapshot_download
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime or record the limitation and use DEVICE='cpu'.")
print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
HF_PATH = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"])
print("Downloaded snapshot:", HF_PATH)

In [ ]:
# Release a previous model before re-running the load measurement.
import gc
if "hf_model" in globals():
    del hf_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_hf():
    global tokenizer, hf_model
    tokenizer = AutoTokenizer.from_pretrained(HF_PATH, local_files_only=True)
    hf_model = AutoModelForCausalLM.from_pretrained(HF_PATH, local_files_only=True).to(DEVICE)
    hf_model.eval()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return {"snapshot": str(HF_PATH), "dtype": str(next(hf_model.parameters()).dtype)}

hf_load = record("transformers", HF_MODEL, 0.135, "", "load", BASE_SETTINGS, load_hf)

def hf_answer(prompt, settings):
    set_seed(settings["seed"])
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = hf_model.generate(**inputs, max_new_tokens=settings["max_tokens"],
                                   do_sample=True, temperature=settings["temperature"],
                                   top_p=settings["top_p"], pad_token_id=tokenizer.eos_token_id)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    tokens = output[0, inputs["input_ids"].shape[1]:]
    return {"answer": tokenizer.decode(tokens, skip_special_tokens=True),
            "output_tokens": len(tokens), "rate_definition": "output tokens / full client generation time"}

if hf_load["status"] == "ok":
    for pid, prompt in PROMPTS.items():
        record("transformers", HF_MODEL, 0.135, pid, "baseline", BASE_SETTINGS,
               lambda prompt=prompt: hf_answer(prompt, BASE_SETTINGS))

## 7. Graphical interface (Tasks 1 and 3)
Launch this small Gradio interface backed by Transformers. Use one of your five prompts and screenshot the prompt, model label, and answer. The UI is for demonstration, separate from benchmark measurements.

Locally/VM: leave sharing disabled; VM users can tunnel port 7860 as well. In Colab, an inline interface may require a Gradio share link. Set `COLAB_PUBLIC_LINK=True` only if you intend to create a temporary public URL; use synthetic prompts only. Stop the interface afterward with `demo.close()`.

In [ ]:
import gradio as gr
COLAB_PUBLIC_LINK = False
if "demo" in globals():
    demo.close()
if "hf_model" not in globals():
    raise RuntimeError("Complete the Transformers loading cell before starting the GUI.")
demo = gr.Interface(
    fn=lambda prompt: hf_answer(prompt, BASE_SETTINGS)["answer"],
    inputs=gr.Textbox(label="Your synthetic test prompt", value=PROMPTS["p1_security"], lines=4),
    outputs=gr.Textbox(label="Model answer", lines=8),
    title=f"IT7075 • {HF_MODEL}",
    description="Hugging Face Transformers hosting with a Gradio graphical interface.")
demo.launch(server_name="127.0.0.1", server_port=7860, share=COLAB_PUBLIC_LINK,
            inline=True, prevent_thread_lock=True)

## 8. Controlled settings experiment (Task 4)
Use the same model, prompt, and seed. Compare temperature **0.1 vs 1.2** while holding the other settings fixed; then compare **top-p 0.9 vs 0.3** at temperature 1.2. Similar answers are a legitimate outcome. Repeat with several seeds if you want stronger evidence; a single sample does not establish a trend.

In [ ]:
SETTINGS_MODEL = "llama3.2:1b"
variants = [dict(BASE_SETTINGS, temperature=0.1),
            dict(BASE_SETTINGS, temperature=1.2),
            dict(BASE_SETTINGS, temperature=1.2, top_p=0.3)]
# Load outside response timing to keep these runs warm.
api_post("/api/generate", {"model": SETTINGS_MODEL, "stream": False, "keep_alive": "10m",
                         "options": ollama_options(BASE_SETTINGS)})
for settings in variants:
    record("ollama", SETTINGS_MODEL, 1.0, "p1_security", "settings", settings,
           lambda settings=settings: ollama_answer(SETTINGS_MODEL, PROMPTS["p1_security"], settings))
api_post("/api/generate", {"model": SETTINGS_MODEL, "keep_alive": 0})

**My observations:** _Which claims, wording, or format changed? Were unsupported claims introduced? What does that imply for analyst work? Quote actual answers._

## 9. Repeat in Colab and Jetstream2
Run sections 1–8 on each machine with **identical prompts and baseline settings**. Keep the prompt-plan JSON and model IDs unchanged. For each environment, capture hardware, model output, the GUI, failures, and your fix attempts. Repeat trials if time permits. Actual GPU placement, quantization, tokenizer, and backend differences should be discussed as confounding factors.

For Jetstream2, the next cell optionally tries a larger model. Record an allocation/memory failure honestly. Do not run a model larger than your available resources simply to force a failure.

In [ ]:
TRY_LARGER_MODEL = False
if TRY_LARGER_MODEL:
    assert ENVIRONMENT == "jetstream2", "This optional experiment is intended for your class VM."
    large_model = "qwen2.5:7b"
    try:
        subprocess.run(["ollama", "pull", large_model], check=True)
        run_ollama(large_model, 7.0)
    except Exception as e:
        def failed_attempt():
            raise RuntimeError(str(e))
        record("ollama", large_model, 7.0, "", "setup", BASE_SETTINGS, failed_attempt)

## 10. Export and compare your evidence
Each environment has a separate result folder. Download its ZIP, then extract all environment folders into one `llmhosting_results` directory on your local machine. Rerun the comparison cell to combine them. Avoid duplicating folders; every execution is retained, including repeated trials.

The chart averages successful warm baseline responses across your prompts. Compare matching model/backend/device configurations and inspect variability; different output lengths can change total latency. Missing environments and failures are not zero-valued measurements. Do not use these averages as a quality score.

In [ ]:
files = list((PROJECT / "llmhosting_results").glob("*/runs.jsonl"))
rows = [json.loads(line) for file in files for line in file.read_text().splitlines() if line.strip()]
df = pd.DataFrame(rows)
if df.empty:
    print("No experiments recorded yet.")
else:
    df.to_csv(OUT / "all_runs.csv", index=False)
    display(df[["environment", "device", "model", "prompt_id", "phase", "load_s", "response_s",
                "tokens_s", "ram_peak_gib", "status"]])
    baseline = df[(df.status == "ok") & (df.phase == "baseline") & (df.plan_id == PLAN_ID)]
    if not baseline.empty:
        summary = baseline.groupby(["environment", "device", "backend", "model"]).agg(
            responses=("response_s", "count"), mean_response_s=("response_s", "mean"),
            std_response_s=("response_s", "std"), mean_tokens_s=("tokens_s", "mean"),
            max_system_ram_gib=("ram_peak_gib", "max"))
        display(summary)
        summary.to_csv(OUT / "comparison.csv")
        import matplotlib.pyplot as plt
        ax = summary["mean_response_s"].plot.barh(figsize=(10, max(3, len(summary)*0.6)),
                    title="Warm response time — recorded baseline runs")
        ax.set_xlabel("Mean client wall time (seconds); lower is faster")
        ax.set_ylabel("Environment / device / backend / model")
        plt.tight_layout()
        plt.savefig(OUT / "response_times.png", dpi=180, bbox_inches="tight")
        plt.show()
    failures = df[df.status == "error"]
    display(failures[["environment", "model", "phase", "error"]])

In [ ]:
archive = shutil.make_archive(str(PROJECT / f"llmhosting-{ENVIRONMENT}"), "zip",
                              root_dir=OUT.parent, base_dir=ENVIRONMENT)
print("Saved:", archive)
# In Colab, download before disconnecting:
if ENVIRONMENT == "colab":
    from google.colab import files
    files.download(archive)

## 11. Your analysis and submission checklist
Fill this table using your observations. If Jetstream2 is waived, state that explicitly rather than inventing results.

| Dimension | Local | Colab | Jetstream2 |
|---|---|---|---|
| Hardware and GPU placement | | | |
| Setup steps, time, and difficulty | | | |
| Speed (cite CSV rows/chart) | | | |
| Successful model sizes | | | |
| Failures and attempted fixes | | | |
| Cost / class allocation usage | | | |
| Privacy and where inference/data reside | | | |
| Reliability / session persistence | | | |

### Recommendations supported by your measurements
1. **Everyday security analyst:** _Choose an environment, cite measured results, and explain quality/speed tradeoffs._
2. **Sensitive or confidential data:** _Choose an environment and explain access controls and data location. Use synthetic data for this assignment._
3. **Quick experiment due today:** _Choose an environment and cite actual setup time, speed, and reliability._

### Troubleshooting journal
| Environment/task | Exact error or symptom | What I changed | Result / what I learned |
|---|---|---|---|
| | | | |

### Evidence checklist
- [ ] Plan and five unchanged prompts, including security and expected-failure prompts.
- [ ] Two hosting engines, with a graphical interface shown working.
- [ ] Three models, two or more families and sizes, one downloaded directly from Hugging Face.
- [ ] Five answers per model; outputs in every required environment.
- [ ] Two interaction methods, showing commands/prompts and answers.
- [ ] Low/high temperature and one other setting, with actual responses and interpretation.
- [ ] Hardware, load time, response time, throughput definition, RAM measurement method, and failures.
- [ ] Tables plus at least one chart; all six comparison dimensions above.
- [ ] Three recommendations backed by your own results.
- [ ] Screenshots labelled by task number, without passwords or personal data.
- [ ] One PDF report with video link on first page; notebook/ZIP are working files, not separate submissions.

**Suggested live video outline (5–10 minutes):** 0–1 min plan and models; 1–3 min local generation and GUI; 3–5 min Colab generation; 5–7 min Jetstream2 generation/larger-model attempt (if required); final 1–2 min chart, surprising result, and what you would change. Use live screens rather than slides of screenshots.

**Cleanup:** Close Gradio below. Download Colab files. Unload Ollama models if needed (`ollama stop MODEL`). Stop the Jetstream2 VM in the cloud dashboard. Keep your results and screenshots until grading is complete.

### Common fixes
- **Connection refused on port 11434:** start `ollama serve`, then retry the health check. If it says the port is already in use, check `ollama list`; another server may already be running.
- **Wrong kernel / missing imports:** select `.venv-llmhosting/bin/python`; installing into another Python environment will not fix this kernel.
- **CUDA unavailable:** verify GPU allocation and PyTorch's CUDA build. Record the issue; do not label a CPU run as GPU.
- **Out of memory:** unload other Ollama models, close the Gradio interface, restart the kernel to release Transformers memory, and retry a smaller model. Record the original failure and the change.
- **Truncated answer:** 128 new tokens is the baseline limit. Keep it fixed for baseline comparisons; test a larger limit as a separately labelled settings experiment.
- **Gradio port already in use:** close the previous interface or change `server_port`; use the matching SSH tunnel port on a VM.


In [ ]:
if "demo" in globals():
    demo.close()
print("Keep your exported results, screenshots, and notes for the report.")